# XGBOOST MODELS FOR CREDIT EVALUATION

## UNCONSTRAINED XGBOOST MODEL

### IMPORTS & DATA

In [5]:
# file-path handling
from pathlib import Path

# Utilities for data management
import numpy as np
import pandas as pd

In [6]:
# Ouput paths for the 3 splits
OUTPUT_DIR = Path("../data")
TRAIN_PATH = OUTPUT_DIR / "train.csv"
VALIDATE_PATH = OUTPUT_DIR / "validate.csv"
TEST_PATH = OUTPUT_DIR / "test.csv"

# Load data splits to pandas (no low mem for better typing when loading to df)
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
validate_df = pd.read_csv(VALIDATE_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)

# Data shape prints (samples, features)
print("Train shape:   ", train_df.shape)
print("Validate shape:", validate_df.shape)
print("Test shape:    ", test_df.shape)

Train shape:    (215137, 33)
Validate shape: (46101, 33)
Test shape:     (46101, 33)


Splits are correct (70% / 15% / 15%)

No. of Features match preprocesing output (33)

In [7]:
# Validate data loaded (after project-level preprocessing - see preprocess-data.ipynb)

# Create resuable constants for label and ID
TARGET_COLUMN = "TARGET"
ID_COLUMN = "SK_ID_CURR"

# Confirm features and order consistent across splits
print("Train/validate columns match:", list(train_df.columns) == list(validate_df.columns))
print("Train/test columns match:    ", list(train_df.columns) == list(test_df.columns))

# Confirm class imbalance (already stratified split)
print("\nDefault rate by split:")
print("Train:   ", train_df[TARGET_COLUMN].mean())
print("Validate:", validate_df[TARGET_COLUMN].mean())
print("Test:    ", test_df[TARGET_COLUMN].mean())

# Check dtypes (esp for categorical features still strings after preprocessing)

# Group for dtype: [list of columns]
dtype_to_columns = (
    train_df.dtypes
    .astype(str)                                # Convert dtype objects to readable strings
    .groupby(train_df.dtypes.astype(str))       # Group columns by dtype
    .apply(lambda s: sorted(s.index.tolist()))  # Collect and sort column names in each dtype group
)

print("\nFeatures by dtype:")
for dtype_name, columns in dtype_to_columns.items():
    print(f"{dtype_name}: {columns}")

Train/validate columns match: True
Train/test columns match:     True

Default rate by split:
Train:    0.08072995347150885
Validate: 0.08073577579662046
Test:     0.08071408429318236

Features by dtype:
float64: ['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'CNT_FAM_MEMBERS', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'OWN_CAR_AGE']
int64: ['CNT_CHILDREN', 'DAYS_BIRTH', 'DAYS_ID_PUBLISH', 'EXT_SOURCE_1_MISSING', 'EXT_SOURCE_2_MISSING', 'EXT_SOURCE_3_MISSING', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'SK_ID_CURR', 'TARGET']
str: ['NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


Cols match
Class imbalance consistent across splits but (!) very pronounced